# Wave Speed in Non-Conservative Form

### Why Scalar Dissipation Fixes Shock Speed for Burgers but NOT for Euler

**Key Insight:** For a *scalar* conservation law (Burgers), adding scalar artificial viscosity to the non-conservative form recovers the correct shock speed. For a *system* (Euler equations), scalar dissipation is insufficient — the dissipation must be a **matrix** that aligns with all wave speeds (Riemann invariants) to capture the correct shock speed.

---

## Mathematical Background

### Conservative vs Non-Conservative Form

A conservation law in **conservative form**:
$$
\frac{\partial \mathbf{U}}{\partial t} + \frac{\partial \mathbf{F}(\mathbf{U})}{\partial x} = 0
$$

In **non-conservative (quasilinear) form**:
$$
\frac{\partial \mathbf{U}}{\partial t} + \mathbf{A}(\mathbf{U})\frac{\partial \mathbf{U}}{\partial x} = 0,
\quad \mathbf{A}(\mathbf{U}) = \frac{\partial \mathbf{F}}{\partial \mathbf{U}}
$$

These are equivalent for smooth solutions. At shocks, the conservative form enforces the **Rankine-Hugoniot jump condition**:
$$
s[\mathbf{U}] = [\mathbf{F}]
$$
where $s$ is the shock speed. The non-conservative form does not enforce this naturally.

### The Path Integral (DLM) Framework

For non-conservative systems, the product $\mathbf{A}(\mathbf{U}) \partial \mathbf{U} / \partial x$ is **not well-defined** at discontinuities. Dal Maso, LeFloch, and Murat (1995) resolved this by defining a family of **paths** $\varphi(s; \mathbf{U}_L, \mathbf{U}_R)$ connecting left and right states. The generalized jump condition becomes a **path integral**:

$$
s[\mathbf{U}] = \int_0^1 \mathbf{A}(\varphi(s; \mathbf{U}_L, \mathbf{U}_R)) \, \frac{\partial \varphi}{\partial s} \, ds
$$

Different path choices yield **different shock speeds**. The physically correct path corresponds to the vanishing-viscosity limit, which for Euler is the straight line in conservative variables ($\partial \mathbf{U}/\partial s = \text{constant}$). This recovers the standard Rankine-Hugoniot condition:

$$
s[\mathbf{U}] = \int_0^1 \frac{\partial \mathbf{F}}{\partial \mathbf{U}} \frac{\partial \mathbf{U}}{\partial s} ds = \int_0^1 \frac{d\mathbf{F}(\varphi)}{ds} ds = [\mathbf{F}]
$$

The **path-conservative** numerical scheme (Abgrall &amp; Karni, 2001) discretizes this path integral directly, ensuring the correct shock speed even in non-conservative form. As shown in Neelan et al. (2026, arXiv:2604.01968), this framework also enables correct shock capturing in non-conservative PINNs via path-integral informed loss functions.

### Scalar Case: Burgers Equation ($u_t + u u_x = 0$)

Conservative form (with viscosity $\nu$):
$$
u_t + \left(\frac{u^2}{2}\right)_x = \nu u_{xx}
$$

Non-conservative form with scalar LF dissipation ($\alpha = \max|u|$):
$$
u_t + u u_x = \nu u_{xx} + \frac{\alpha \Delta x}{2} u_{xx}
$$

For a scalar equation, there is only **one** wave speed ($u$). The scalar dissipation $\frac{\alpha \Delta x}{2} u_{xx}$ adds diffusion that stabilizes the shock at the correct speed. The single wave speed means scalar dissipation can align with the characteristic.

### System Case: Euler Equations

The Euler equations in conservative form:
$$
\frac{\partial}{\partial t}\begin{bmatrix}\rho \\ \rho u \\ E\end{bmatrix} +
\frac{\partial}{\partial x}\begin{bmatrix}\rho u \\ \rho u^2 + p \\ (E+p)u\end{bmatrix} = 0
$$

Non-conservative form with scalar dissipation:
$$
\mathbf{U}_t + \mathbf{A}(\mathbf{U})\mathbf{U}_x = \nu \mathbf{U}_{xx},
\quad \nu = \frac{\alpha \Delta x}{2},
\quad \alpha = \max(|u|+c)
$$

The Jacobian $\mathbf{A}$ has **three** distinct eigenvalues:
$$
\lambda_1 = u-c,\quad \lambda_2 = u,\quad \lambda_3 = u+c
$$

Scalar dissipation applies the **same** diffusion to all characteristic fields. But each wave travels at a different speed! To get the correct shock speed, we need **matrix dissipation** that applies different amounts of diffusion to each characteristic field:
$$
\mathbf{U}_t + \mathbf{A}\mathbf{U}_x = \mathbf{R}\,|\boldsymbol{\Lambda}|\,\mathbf{R}^{-1}\frac{\Delta x}{2}\mathbf{U}_{xx}
$$
where $\mathbf{R}$ contains the right eigenvectors of $\mathbf{A}$, and $|\boldsymbol{\Lambda}| = \text{diag}(|\lambda_1|, |\lambda_2|, |\lambda_3|)$. This is exactly what the Rusanov flux does in the conservative scheme.

---

## Demos Overview

| Case | Equation | Formulation | Matches Reference? |
|------|----------|-------------|-------------------|
| 1 | Burgers (scalar) | **Conservative** (Rusanov flux) | ✅ Yes |
| 2 | Burgers (scalar) | **Non-conservative** + LF scalar dissipation | ✅ Yes (works for scalar) |
| 3 | Euler (system) | **Conservative** (Rusanov flux) | ✅ Yes |
| 4 | Euler (system) | **Non-conservative** + scalar diffusion | ❌ No (fails for system) |
| 5 | Euler (system) | **Path-Conservative** (Abgrall &amp; Karni) | ✅ Yes (the cure!) |

&gt; This tutorial accompanies the paper: Neelan et al., *Revisiting Conservativeness in Fluid Dynamics: Failure of Non-Conservative PINNs and a Path-Integral Remedy*, arXiv:2604.01968 (2026).

In [ ]:
# %%
import numpy as np
import sympy as sp
from sympy.utilities.lambdify import lambdify
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec

---
## Demo 1: Burgers Equation — Conservative Form (Rusanov)

We solve the viscous Burgers equation using the conservative Rusanov (local Lax-Friedrichs) flux. The analytical solution is obtained via the Cole-Hopf transformation.

In [ ]:
# %%
# --- Burgers: Analytical solution (Cole-Hopf) ---
x_sym, nu_sym, t_sym = sp.symbols('x nu t')
phi = (sp.exp(-(x_sym - 4 * t_sym)**2 / (4 * nu_sym * (t_sym + 1))) +
       sp.exp(-(x_sym - 4 * t_sym - 2 * sp.pi)**2 / (4 * nu_sym * (t_sym + 1))))
phiprime = phi.diff(x_sym)
u_expr = -2 * nu_sym * (phiprime / phi) + 4
ufunc = lambdify((t_sym, x_sym, nu_sym), u_expr)

# Grid
nx = 101; L = 2 * np.pi; dx = L / (nx - 1); nu_val = 0.07
x = np.linspace(0, L, nx)

# Initial condition
t0 = 0.0
u = np.array([ufunc(t0, xi, nu_val) for xi in x])

# Time step
dt = min(0.2 * dx / np.max(np.abs(u)), 0.2 * dx**2 / nu_val)
nt = 50; t_final = nt * dt
print(f"dt = {dt:.5f}")

# --- Conservative time marching (Rusanov) ---
for n in range(nt):
    un = u.copy()
    f = 0.5 * un**2
    alpha = np.max(np.abs(un))
    flux = 0.5 * (f[1:] + f[:-1]) - 0.5 * alpha * (un[1:] - un[:-1])
    u[1:-1] = (un[1:-1]
        - dt/dx * (flux[1:] - flux[:-1])
        + nu_val * dt/dx**2 * (un[2:] - 2*un[1:-1] + un[:-2]))
    # Periodic BC
    f0 = 0.5 * un[0]**2; fL = 0.5 * un[-2]**2
    flux_left = 0.5 * (f0 + fL) - 0.5 * alpha * (un[0] - un[-2])
    flux_right = flux[0]
    u[0] = (un[0]
        - dt/dx * (flux_right - flux_left)
        + nu_val * dt/dx**2 * (un[1] - 2*un[0] + un[-2]))
    u[-1] = u[0]

# Analytical solution at final time
u_analytical = np.array([ufunc(t_final, xi, nu_val) for xi in x])

# Plot
plt.figure(figsize=(10, 6), dpi=100)
plt.plot(x, u, 'o-', lw=2, label='Rusanov (Conservative)')
plt.plot(x, u_analytical, '--', lw=2, label='Analytical (Cole-Hopf)')
plt.xlim([0, L]); plt.ylim([0, 10])
plt.xlabel('x'); plt.ylabel('u')
plt.title('Burgers: Conservative Form (Rusanov) — Matches Analytical')
plt.legend(); plt.grid(); plt.show()

---
## Demo 2: Burgers Equation — Non-Conservative Form + LF Scalar Dissipation

Here we solve $u_t + u u_x = \nu u_{xx} + \frac{\alpha \Delta x}{2} u_{xx}$. The scalar dissipation term (last term) is added to stabilize the shock. **For a scalar equation, this works because there is only one wave speed.**

In [ ]:
# %%
# Reset initial condition
u = np.array([ufunc(t0, xi, nu_val) for xi in x])

# --- Non-conservative time marching + LF scalar dissipation ---
for n in range(nt):
    un = u.copy()
    alpha = np.max(np.abs(un))
    # Interior: non-conservative advection + physical viscosity + LF dissipation
    u[1:-1] = (un[1:-1]
        - un[1:-1] * dt/dx * (un[1:-1] - un[:-2])
        + nu_val * dt/dx**2 * (un[2:] - 2*un[1:-1] + un[:-2])
        + (alpha * dt / (2*dx)) * (un[2:] - 2*un[1:-1] + un[:-2]))
    # Periodic BC
    u[0] = (un[0]
        - un[0] * dt/dx * (un[0] - un[-2])
        + nu_val * dt/dx**2 * (un[1] - 2*un[0] + un[-2])
        + (alpha * dt / (2*dx)) * (un[1] - 2*un[0] + un[-2]))
    u[-1] = u[0]

u_analytical = np.array([ufunc(t_final, xi, nu_val) for xi in x])

plt.figure(figsize=(10, 6), dpi=100)
plt.plot(x, u, 'b--', lw=2, label='Non-conservative + LF dissipation')
plt.plot(x, u_analytical, 'k-', lw=2, label='Analytical (Cole-Hopf)')
plt.xlim([0, L]); plt.ylim([0, 10])
plt.xlabel('x'); plt.ylabel('u')
plt.title('Burgers: Non-Conservative + LF Scalar Dissipation — Also Matches!')
plt.legend(); plt.grid(); plt.show()

---
## Demo 3: Euler Equations — Conservative Form (Rusanov)

We solve the Sod shock tube problem using the conservative Rusanov scheme. The exact Riemann solution is computed for comparison.

In [ ]:
# %%
# --- Euler: Parameters and helpers ---
gamma = 1.4; L_e = 2.0; Nx = 1000
x_e = np.linspace(0, L_e, Nx); dx_e = x_e[1] - x_e[0]
T_e = 0.5; CFL = 0.5; x0 = 1.0

# Initial condition (Sod problem)
rho0 = np.where(x_e < x0, 1.0, 0.125)
u0_e = np.zeros_like(x_e)
p0   = np.where(x_e < x0, 1.0, 0.1)

def prim_to_cons(rho, u, p):
    E = p/(gamma-1) + 0.5*rho*u**2
    return np.vstack((rho, rho*u, E)).T

def cons_to_prim(U):
    rho = U[:,0]; u = U[:,1]/rho; E = U[:,2]
    p = (gamma-1)*(E - 0.5*rho*u**2)
    return rho, u, p

# --- Exact Riemann Solver ---
def exact_sod(x, t, gamma=1.4):
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1
    if t == 0.0:
        rho = np.where(x < x0, rhoL, rhoR)
        u   = np.zeros_like(x)
        p   = np.where(x < x0, pL, pR)
        return prim_to_cons(rho, u, p)
    cL = math.sqrt(gamma * pL / rhoL)
    cR = math.sqrt(gamma * pR / rhoR)
    def f(p, rho, p0, c):
        if p > p0:
            A = 2 / ((gamma + 1) * rho); B = (gamma - 1) / (gamma + 1) * p0
            return (p - p0) * math.sqrt(A / (p + B))
        else:
            return (2 * c / (gamma - 1)) * ((p / p0)**((gamma - 1)/(2*gamma)) - 1)
    def df(p, rho, p0, c):
        if p > p0:
            A = 2 / ((gamma + 1) * rho); B = (gamma - 1) / (gamma + 1) * p0
            return math.sqrt(A / (p + B)) * (1 - 0.5 * (p - p0)/(p + B))
        else:
            return (c / (gamma * p0)) * (p / p0)**(-(gamma + 1)/(2*gamma))
    p = max(1e-6, 0.5*(pL+pR) - 0.125*(uR-uL)*(rhoL+rhoR)*(cL+cR))
    for _ in range(20):
        p -= (f(p,rhoL,pL,cL)+f(p,rhoR,pR,cR)+uR-uL)/(df(p,rhoL,pL,cL)+df(p,rhoR,pR,cR))
    p_star = p
    u_star = 0.5*(uL+uR) + 0.5*(f(p_star,rhoR,pR,cR)-f(p_star,rhoL,pL,cL))
    rho=np.zeros_like(x); u=np.zeros_like(x); p=np.zeros_like(x)
    for i in range(len(x)):
        xi = (x[i]-x0)/t
        if xi < u_star:
            if not (p_star > pL):
                shl = uL-cL
                stl = u_star - math.sqrt(gamma*p_star/(rhoL*(p_star/pL)**(1/gamma)))
                if xi < shl:
                    rho[i],u[i],p[i]=rhoL,uL,pL
                elif xi < stl:
                    u[i]=(2/(gamma+1))*(cL+xi)
                    c=cL-0.5*(gamma-1)*u[i]
                    rho[i]=rhoL*(c/cL)**(2/(gamma-1))
                    p[i]=pL*(c/cL)**(2*gamma/(gamma-1))
                else:
                    rho[i]=rhoL*(p_star/pL)**(1/gamma); u[i]=u_star; p[i]=p_star
        else:
            if p_star > pR:
                sr = uR + cR*math.sqrt((gamma+1)/(2*gamma)*(p_star/pR)+(gamma-1)/(2*gamma))
                if xi > sr:
                    rho[i],u[i],p[i]=rhoR,uR,pR
                else:
                    rho[i]=rhoR*((p_star/pR+(gamma-1)/(gamma+1))/((gamma-1)/(gamma+1)*p_star/pR+1))
                    u[i]=u_star; p[i]=p_star
    return prim_to_cons(rho,u,p)

In [ ]:
# %%
# --- Euler Conservative (Rusanov) time marching ---
rho_rus = rho0.copy(); u_rus = u0_e.copy(); p_rus = p0.copy()
t_e = 0.0

rho_rus_frames = [rho0.copy()]
time_list_e = [0.0]

while t_e < T_e:
    a_rus = np.sqrt(gamma*p_rus/rho_rus)
    dt_e = CFL*dx_e/np.max(np.abs(u_rus)+a_rus)
    if t_e + dt_e > T_e:
        dt_e = T_e - t_e
    
    U = prim_to_cons(rho_rus, u_rus, p_rus)
    rho, u, p = cons_to_prim(U)
    a = np.sqrt(gamma * p / rho)
    F = np.zeros_like(U)
    F[:,0]=rho*u; F[:,1]=rho*u**2+p; F[:,2]=(U[:,2]+p)*u
    F_half = np.zeros((Nx-1,3))
    for i in range(Nx-1):
        alpha = max(abs(u[i])+a[i], abs(u[i+1])+a[i+1])
        F_half[i] = 0.5*(F[i]+F[i+1]) - 0.5*alpha*(U[i+1]-U[i])
    U_new = U.copy()
    for i in range(1,Nx-1):
        U_new[i] = U[i] - dt_e/dx_e*(F_half[i]-F_half[i-1])
    U_new[0]=U_new[1]; U_new[-1]=U_new[-2]
    rho_rus, u_rus, p_rus = cons_to_prim(U_new)
    t_e += dt_e
    rho_rus_frames.append(rho_rus.copy())
    time_list_e.append(t_e)

U_exact = exact_sod(x_e, t_e, gamma)
rho_exact = U_exact[:,0]

plt.figure(figsize=(10, 6), dpi=100)
plt.plot(x_e, rho_rus, 'r.', markersize=3, label='Conservative (Rusanov)', markevery=5)
plt.plot(x_e, rho_exact, 'k-', lw=2, label='Exact Riemann')
plt.xlim([0, L_e]); plt.ylim([0, 2.1])
plt.xlabel('x'); plt.ylabel(r'$\rho$')
plt.title('Euler: Conservative Form (Rusanov) — Matches Exact')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

---
## Demo 4: Euler Equations — Non-Conservative Form + Scalar Diffusion

Here we solve the non-conservative form with scalar diffusion applied equally to all equations:
$$
\mathbf{U}_t + \mathbf{A}\mathbf{U}_x = \nu \mathbf{U}_{xx},
\quad \nu = \frac{\alpha \Delta x}{2}
$$

**Result:** The shock is at the wrong location! Scalar dissipation cannot correctly capture all three wave speeds in the Euler system.

In [ ]:
# %%
# --- Euler Non-Conservative + Scalar Diffusion ---
rho_non = rho0.copy(); u_non = u0_e.copy(); p_non = p0.copy()
t_e = 0.0

rho_non_frames = [rho0.copy()]
time_list_nc = [0.0]

while t_e < T_e:
    a_non = np.sqrt(gamma*p_non/rho_non)
    dt_e = CFL*dx_e/np.max(np.abs(u_non)+a_non)
    if t_e + dt_e > T_e:
        dt_e = T_e - t_e

    rho=rho_non; u=u_non; p=p_non
    rho_x=(rho[2:]-rho[:-2])/(2*dx_e); u_x=(u[2:]-u[:-2])/(2*dx_e); p_x=(p[2:]-p[:-2])/(2*dx_e)
    rho_xx=(rho[2:]-2*rho[1:-1]+rho[:-2])/dx_e**2
    u_xx=(u[2:]-2*u[1:-1]+u[:-2])/dx_e**2
    p_xx=(p[2:]-2*p[1:-1]+p[:-2])/dx_e**2
    alpha_non = np.max(np.abs(u)+np.sqrt(gamma*p/rho))
    nu = 0.5*alpha_non*dx_e
    
    rho_new=rho.copy(); u_new=u.copy(); p_new=p.copy()
    rho_new[1:-1]=rho[1:-1]-dt_e*(u[1:-1]*rho_x+rho[1:-1]*u_x)+dt_e*nu*rho_xx
    u_new[1:-1]=u[1:-1]-dt_e*(u[1:-1]*u_x+p_x/rho[1:-1])+dt_e*nu*u_xx
    p_new[1:-1]=p[1:-1]-dt_e*(u[1:-1]*p_x+gamma*p[1:-1]*u_x)+dt_e*nu*p_xx
    rho_new[0]=rho_new[1]; rho_new[-1]=rho_new[-2]
    u_new[0]=u_new[1]; u_new[-1]=u_new[-2]
    p_new[0]=p_new[1]; p_new[-1]=p_new[-2]
    
    rho_non, u_non, p_non = rho_new, u_new, p_new
    t_e += dt_e
    rho_non_frames.append(rho_non.copy())
    time_list_nc.append(t_e)

U_exact = exact_sod(x_e, t_e, gamma)
rho_exact = U_exact[:,0]

plt.figure(figsize=(10, 6), dpi=100)
plt.plot(x_e, rho_non, 'b--', lw=2, label='Non-Conservative + Scalar Diffusion')
plt.plot(x_e, rho_exact, 'k-', lw=2, label='Exact Riemann')
plt.xlim([0, L_e]); plt.ylim([0, 2.1])
plt.xlabel('x'); plt.ylabel(r'$\rho$')
plt.title('Euler: Non-Conservative + Scalar Diffusion — WRONG Shock Speed!')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

---
## Combined 2×2 Animation

The animation below shows all four cases simultaneously. Watch how:
- **Top row:** Both Burgers schemes match the analytical solution
- **Bottom left:** Conservative Euler matches exact Riemann
- **Bottom right:** Non-conservative Euler + scalar diffusion gives the **wrong shock location**

This clearly demonstrates that scalar dissipation is sufficient for scalar equations but fails for systems.

In [ ]:
# %%
# --- Re-run Burgers with frame storage ---
u_burgers_exact_ic = np.array([ufunc(t0, xi, nu_val) for xi in x])
dt_b = min(0.2 * dx / np.max(np.abs(u_burgers_exact_ic)), 0.2 * dx**2 / nu_val)
nt_b = 50

u_cons_frames_b = [u_burgers_exact_ic.copy()]
u_non_frames_b = [u_burgers_exact_ic.copy()]
u_exact_frames_b = [u_burgers_exact_ic.copy()]
time_b = [0.0]

u_cons_b = u_burgers_exact_ic.copy()
u_non_b = u_burgers_exact_ic.copy()
t_b = 0.0

for n in range(nt_b):
    t_b += dt_b
    # Conservative
    un = u_cons_b.copy()
    f = 0.5 * un**2
    alpha = np.max(np.abs(un))
    flux = 0.5*(f[1:]+f[:-1]) - 0.5*alpha*(un[1:]-un[:-1])
    u_cons_b[1:-1] = (un[1:-1] - dt_b/dx_b*(flux[1:]-flux[:-1])
        + nu_val*dt_b/dx_b**2*(un[2:]-2*un[1:-1]+un[:-2]))
    f0=0.5*un[0]**2; fL=0.5*un[-2]**2
    flux_left=0.5*(f0+fL)-0.5*alpha*(un[0]-un[-2])
    flux_right=flux[0]
    u_cons_b[0]=(un[0]-dt_b/dx_b*(flux_right-flux_left)
        + nu_val*dt_b/dx_b**2*(un[1]-2*un[0]+un[-2]))
    u_cons_b[-1]=u_cons_b[0]
    # Non-conservative
    un = u_non_b.copy()
    alpha = np.max(np.abs(un))
    u_non_b[1:-1] = (un[1:-1]-un[1:-1]*dt_b/dx_b*(un[1:-1]-un[:-2])
        + nu_val*dt_b/dx_b**2*(un[2:]-2*un[1:-1]+un[:-2])
        + (alpha*dt_b/(2*dx_b))*(un[2:]-2*un[1:-1]+un[:-2]))
    u_non_b[0]=(un[0]-un[0]*dt_b/dx_b*(un[0]-un[-2])
        + nu_val*dt_b/dx_b**2*(un[1]-2*un[0]+un[-2])
        + (alpha*dt_b/(2*dx_b))*(un[1]-2*un[0]+un[-2]))
    u_non_b[-1]=u_non_b[0]
    # Exact
    u_exact_b = np.array([ufunc(t_b, xi, nu_val) for xi in x])
    u_cons_frames_b.append(u_cons_b.copy())
    u_non_frames_b.append(u_non_b.copy())
    u_exact_frames_b.append(u_exact_b)
    time_b.append(t_b)

In [ ]:
# %%
# --- Re-run Euler with frame storage (both schemes) ---
rho_rus = rho0.copy(); u_rus = u0_e.copy(); p_rus = p0.copy()
rho_non = rho0.copy(); u_non = u0_e.copy(); p_non = p0.copy()
t_e = 0.0

rho_rus_frames2 = [rho0.copy()]
rho_non_frames2 = [rho0.copy()]
rho_exact_frames2 = [exact_sod(x_e, 0.0, gamma)[:,0]]
time_list_e2 = [0.0]

while t_e < T_e:
    a_rus=np.sqrt(gamma*p_rus/rho_rus); a_non=np.sqrt(gamma*p_non/rho_non)
    max_speed = max(np.max(np.abs(u_rus)+a_rus), np.max(np.abs(u_non)+a_non))
    dt_e = CFL*dx_e/max_speed
    if t_e+dt_e > T_e: dt_e = T_e-t_e

    # Conservative
    U=prim_to_cons(rho_rus,u_rus,p_rus)
    rho,u,p=cons_to_prim(U)
    a=np.sqrt(gamma*p/rho)
    F=np.zeros_like(U); F[:,0]=rho*u; F[:,1]=rho*u**2+p; F[:,2]=(U[:,2]+p)*u
    F_half=np.zeros((Nx-1,3))
    for i in range(Nx-1):
        alpha=max(abs(u[i])+a[i],abs(u[i+1])+a[i+1])
        F_half[i]=0.5*(F[i]+F[i+1])-0.5*alpha*(U[i+1]-U[i])
    U_new=U.copy()
    for i in range(1,Nx-1):
        U_new[i]=U[i]-dt_e/dx_e*(F_half[i]-F_half[i-1])
    U_new[0]=U_new[1]; U_new[-1]=U_new[-2]
    rho_rus,u_rus,p_rus=cons_to_prim(U_new)

    # Non-conservative
    r=rho_non; u=u_non; p=p_non
    rho_x=(r[2:]-r[:-2])/(2*dx_e); u_x=(u[2:]-u[:-2])/(2*dx_e); p_x=(p[2:]-p[:-2])/(2*dx_e)
    rho_xx=(r[2:]-2*r[1:-1]+r[:-2])/dx_e**2
    u_xx=(u[2:]-2*u[1:-1]+u[:-2])/dx_e**2
    p_xx=(p[2:]-2*p[1:-1]+p[:-2])/dx_e**2
    alpha_non=np.max(np.abs(u)+np.sqrt(gamma*p/r))
    nu=0.5*alpha_non*dx_e
    rho_new=r.copy(); u_new=u.copy(); p_new=p.copy()
    rho_new[1:-1]=r[1:-1]-dt_e*(u[1:-1]*rho_x+r[1:-1]*u_x)+dt_e*nu*rho_xx
    u_new[1:-1]=u[1:-1]-dt_e*(u[1:-1]*u_x+p_x/r[1:-1])+dt_e*nu*u_xx
    p_new[1:-1]=p[1:-1]-dt_e*(u[1:-1]*p_x+gamma*p[1:-1]*u_x)+dt_e*nu*p_xx
    rho_new[0]=rho_new[1]; rho_new[-1]=rho_new[-2]
    u_new[0]=u_new[1]; u_new[-1]=u_new[-2]
    p_new[0]=p_new[1]; p_new[-1]=p_new[-2]
    rho_non,u_non,p_non=rho_new,u_new,p_new

    t_e+=dt_e
    U_exact_t=exact_sod(x_e,t_e,gamma); rho_exact_t=U_exact_t[:,0]
    rho_rus_frames2.append(rho_rus.copy())
    rho_non_frames2.append(rho_non.copy())
    rho_exact_frames2.append(rho_exact_t)
    time_list_e2.append(t_e)

---
## Demo 5: Euler Equations — Path-Conservative Scheme (Abgrall &amp; Karni)

### The Path Integral Connection

The DLM path integral framework tells us that the non-conservative product must be interpreted through a **path** connecting left and right states. The path-conservative scheme discretizes this directly. For the Euler equations in Lagrangian coordinates $\mathbf{W} = [v, u, \varepsilon]^T$ (specific volume, velocity, internal energy):

$$
\frac{\partial \mathbf{W}}{\partial t} + \mathbf{A}(\mathbf{W})\frac{\partial \mathbf{W}}{\partial m} = 0,
\quad \mathbf{A} = \begin{pmatrix}
0 &amp; -1 &amp; 0 \\
-c^2 &amp; 0 &amp; 1 \\
0 &amp; p &amp; 0
\end{pmatrix}
$$

The path-conservative numerical flux at each interface implements the correct path integral:

$$
\mathbf{D}^{\pm} = \frac{1}{2}\left(\mathbf{B} \pm \mathbf{R}|\boldsymbol{\Lambda}|\mathbf{R}^{-1} \Delta \mathbf{W}\right)
$$

where $\mathbf{B}$ contains the non-conservative product (the central flux), and $\mathbf{R}|\boldsymbol{\Lambda}|\mathbf{R}^{-1}\Delta\mathbf{W}$ provides **matrix-valued** dissipation aligned with characteristic fields. This is fundamentally different from scalar dissipation — each wave ($u-c$, $u$, $u+c$) receives precisely the right amount of diffusion.

### Implementation

The scheme is implemented in Lagrangian mass coordinates. The key steps at each cell interface:
1. Compute averaged states $\bar{v}, \bar{p}$ and sound speed $\bar{c}$
2. Compute jumps $\Delta u, \Delta p$
3. Decompose into characteristic fields: $\mathbf{R}^{-1}\Delta\mathbf{W}$
4. Apply dissipation per field: $|\boldsymbol{\Lambda}| \mathbf{R}^{-1}\Delta\mathbf{W}$
5. Reconstruct: $\mathbf{D}^{\pm} = \frac{1}{2}(\mathbf{B} \pm \mathbf{R}|\boldsymbol{\Lambda}|\mathbf{R}^{-1}\Delta\mathbf{W})$

This is exactly the path integral discretization recommended by the DLM theory and analyzed in Neelan et al. (2026, arXiv:2604.01968).

In [ ]:
# %%
# --- Path-Conservative (Abgrall & Karni) scheme ---
def solve_path_conservative_frames(T=0.5):
    N = 1000; m_max = 1.125; dm = m_max / N
    m = np.linspace(dm/2, m_max - dm/2, N)
    gamma = 1.4
    v = np.where(m < 1.0, 1.0, 8.0)
    u = np.zeros(N)
    p = np.where(m < 1.0, 1.0, 0.1)
    eps = p * v / (gamma - 1)
    W = np.vstack((v, u, eps))
    t = 0.0; CFL = 0.5

    x = np.zeros(N)
    x[0] = v[0] * dm / 2
    for i in range(1, N):
        x[i] = x[i-1] + 0.5 * (v[i-1] + v[i]) * dm
    rho = 1.0 / v

    # Interpolate to uniform grid
    rho_frames_uniform = [np.interp(x_e, x, rho, left=rho[0], right=rho[-1])]
    time_list_pc = [0.0]

    while t < T:
        v, u, eps = W[0, :], W[1, :], W[2, :]
        p = (gamma - 1) * eps / v
        C = np.sqrt(gamma * p / v)
        dt = CFL * dm / np.max(C)
        if t + dt > T: dt = T - t

        v_L,v_R = v[:-1],v[1:]; u_L,u_R = u[:-1],u[1:]; p_L,p_R = p[:-1],p[1:]
        v_bar,p_bar = 0.5*(v_L+v_R),0.5*(p_L+p_R)
        C_bar = np.sqrt(gamma * p_bar / v_bar)
        du, dp = u_R-u_L, p_R-p_L

        absA_dW_1 = -dp / C_bar
        absA_dW_2 = C_bar * du
        absA_dW_3 = (p_bar / C_bar) * dp

        B1,B2,B3 = -du, dp, p_bar*du

        D_minus_1,D_plus_1 = 0.5*(B1-absA_dW_1),0.5*(B1+absA_dW_1)
        D_minus_2,D_plus_2 = 0.5*(B2-absA_dW_2),0.5*(B2+absA_dW_2)
        D_minus_3,D_plus_3 = 0.5*(B3-absA_dW_3),0.5*(B3+absA_dW_3)

        W[0,1:-1] -= (dt/dm)*(D_minus_1[1:]+D_plus_1[:-1])
        W[1,1:-1] -= (dt/dm)*(D_minus_2[1:]+D_plus_2[:-1])
        W[2,1:-1] -= (dt/dm)*(D_minus_3[1:]+D_plus_3[:-1])
        W[:,0],W[:,-1] = W[:,1],W[:,-2]
        t += dt

        # Map to physical x and interpolate
        v, u, eps = W[0,:],W[1,:],W[2,:]
        p = (gamma-1)*eps/v
        x = np.zeros(N)
        x[0] = v[0]*dm/2
        for i in range(1,N):
            x[i] = x[i-1] + 0.5*(v[i-1]+v[i])*dm
        rho = 1.0/v
        rho_uniform = np.interp(x_e, x, rho, left=rho[0], right=rho[-1])
        rho_frames_uniform.append(rho_uniform)
        time_list_pc.append(t)

    return rho_frames_uniform, time_list_pc

rho_pc_frames, time_list_pc = solve_path_conservative_frames(T_e)
print(f'Path-Conservative: {len(time_list_pc)} frames, t_final={time_list_pc[-1]:.4f}')

# Final time comparison
U_exact_final = exact_sod(x_e, time_list_pc[-1], gamma)
rho_exact_final = U_exact_final[:,0]

plt.figure(figsize=(10, 6), dpi=100)
plt.plot(x_e, rho_pc_frames[-1], 'm-', lw=2, label='Path-Conservative (Abgrall &amp; Karni)')
plt.plot(x_e, rho_exact_final, 'k-', lw=2, label='Exact Riemann')
plt.xlim([0, L_e]); plt.ylim([0, 2.1])
plt.xlabel('x'); plt.ylabel(r'$\rho$')
plt.title('Euler: Path-Conservative Scheme — Correct Shock Speed! ✓')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

---
## Conclusion

| Equation | Conservative (Rusanov) | Non-Cons. + Scalar Diss. | Path-Conservative (Abgrall &amp; Karni) |
|----------|----------------------|--------------------------|--------------------------------------|
| **Burgers** (scalar) | ✅ Correct shock speed | ✅ Correct (works for scalar) | N/A |
| **Euler** (system) | ✅ Correct shock speed | ❌ Wrong shock speed | ✅ Correct shock speed ✓ |

### Why?

- **Burgers** has a single wave speed $u$. Scalar dissipation $\frac{\alpha\Delta x}{2}u_{xx}$ adds diffusion at the same rate for the only characteristic field, stabilizing the shock at the correct speed.

- **Euler** has three wave speeds $\lambda = u-c, u, u+c$. Scalar dissipation applies the **same** diffusion to all three equations. To get the correct shock speed, the dissipation must be a **matrix** that applies different amounts of diffusion to each characteristic field:
  $$
  \text{Correct: } \mathbf{U}_t + \mathbf{A}\mathbf{U}_x = \mathbf{R}|\boldsymbol{\Lambda}|\mathbf{R}^{-1}\frac{\Delta x}{2}\mathbf{U}_{xx}
  $$
  This is exactly what the conservative Rusanov scheme does implicitly through its flux formulation.

### The Path-Conservative Cure

The **path-conservative scheme** (Abgrall &amp; Karni, 2001; Neelan et al., 2026) solves the non-conservative.

**Key Takeaway:** For systems of conservation laws, non-conservative formulations with simple scalar dissipation **cannot** capture the correct shock speed. However, with a path-conservative approach (characteristic-based matrix dissipation), non-conservative formulations can be successfully used even for transient, high-speed flows.

---

## References

- Neelan, A. G., Don Bosco, F. S., Jarugumalli, N. S., & Vedarethinam, S. B. (2026). *Revisiting Conservativeness in Fluid Dynamics: Failure of Non-Conservative PINNs and a Path-Integral Remedy*. arXiv:2604.01968. [https://arxiv.org/abs/2604.01968](https://arxiv.org/abs/2604.01968)


- Abgrall, R., & Karni, S. (2001). *Computations of Compressible Multifluids*. Journal of Computational Physics, 169(2), 594-623.